In [1]:
"""
    spline_cubico_natural(x::Vector, y::Vector)

Calcula los coeficientes del spline cúbico natural S(x).
S_j(x) = a_j + b_j(x - x_j) + c_j(x - x_j)^2 + d_j(x - x_j)^3

Argumentos:
- `x`: Vector de nodos x_0, ..., x_n
- `y`: Vector de valores f(x_0), ..., f(x_n) (corresponde a 'a' en el algoritmo)

Retorna:
- `a`, `b`, `c`, `d`: Vectores de coeficientes.
"""
function spline_cubico_natural(x::Vector{T}, y::Vector{T}) where T <: AbstractFloat
    n_puntos = length(x)
    n = n_puntos - 1 # 'n' en el libro es el último índice (cantidad de intervalos)
    
    # Verificación de dimensiones
    if length(y) != n_puntos
        error("Los vectores x e y deben tener la misma longitud.")
    end

    # Inicialización de arreglos
    # a, b, c, d, h, alpha, l, mu, z
    # Ajustamos tamaños para índices 1-based
    a = copy(y) # Paso de entrada: a_i = f(x_i)
    b = zeros(T, n)
    c = zeros(T, n + 1) # Necesitamos c_{n} en el paso 6
    d = zeros(T, n)
    h = zeros(T, n)
    alpha = zeros(T, n) # Solo usamos índices 2 a n (que son 1 a n-1 en libro)
    l = zeros(T, n + 1)
    mu = zeros(T, n + 1)
    z = zeros(T, n + 1)

    # --- Paso 1 ---
    # Calcular pasos h_i = x_{i+1} - x_i
    # Libro: i = 0...n-1  => Julia: i = 1...n
    for i in 1:n
        h[i] = x[i+1] - x[i]
    end

    # --- Paso 2 ---
    # Calcular alpha_i
    # Libro: i = 1...n-1 => Julia: i = 2...n
    for i in 2:n
        term1 = (3 / h[i]) * (a[i+1] - a[i])
        term2 = (3 / h[i-1]) * (a[i] - a[i-1])
        alpha[i] = term1 - term2
    end

    # --- Paso 3 ---
    # Resolver sistema tridiagonal (condiciones de frontera natural)
    # Libro: l_0 = 1, mu_0 = 0, z_0 = 0
    # Julia: índice 1
    l[1] = 1.0
    mu[1] = 0.0
    z[1] = 0.0

    # --- Paso 4 ---
    # Resolver sistema tridiagonal parte intermedia
    # Libro: i = 1...n-1 => Julia: i = 2...n
    for i in 2:n
        l[i] = 2 * (x[i+1] - x[i-1]) - h[i-1] * mu[i-1]
        mu[i] = h[i] / l[i]
        z[i] = (alpha[i] - h[i-1] * z[i-1]) / l[i]
    end

    # --- Paso 5 ---
    # Frontera final
    # Libro: l_n = 1, z_n = 0, c_n = 0
    # Julia: índice n+1
    l[n+1] = 1.0
    z[n+1] = 0.0
    c[n+1] = 0.0

    # --- Paso 6 ---
    # Sustitución hacia atrás para hallar c, b, d
    # Libro: j = n-1, n-2, ..., 0
    # Julia: j = n, n-1, ..., 1
    for j in n:-1:1
        c[j] = z[j] - mu[j] * c[j+1]
        b[j] = (a[j+1] - a[j]) / h[j] - (h[j] * (c[j+1] + 2 * c[j])) / 3
        d[j] = (c[j+1] - c[j]) / (3 * h[j])
    end

    # --- Paso 7 ---
    # Salida: a, b, c, d
    # Nota: 'a' ya contiene los valores f(x_i). 
    # Devolvemos solo la parte útil para los intervalos (1 a n en Julia, 0 a n-1 en libro)
    # Sin embargo, 'c' tiene n+1 elementos. El algoritmo dice retornar j=0..n-1.
    return a[1:n], b, c[1:n], d
end

"""
    evaluar_spline(x_val, x_nodos, a, b, c, d)

Evalúa el spline construido en un punto x_val dado.
Encuentra el subintervalo [x_j, x_{j+1}] y aplica el polinomio cúbico correspondiente.
"""
function evaluar_spline(x_val, x_nodos, a, b, c, d)
    n = length(x_nodos) - 1
    
    # 1. Encontrar el intervalo j tal que x_j <= x_val <= x_{j+1}
    # Por simplicidad usamos búsqueda lineal, para optimizar usar búsqueda binaria.
    j = n # Default al último intervalo si se pasa (o es el borde)
    for i in 1:n
        if x_val <= x_nodos[i+1]
            j = i
            break
        end
    end
    
    # 2. Evaluar el polinomio
    # S_j(x) = a_j + b_j(dx) + c_j(dx)^2 + d_j(dx)^3
    dx = x_val - x_nodos[j]
    val = a[j] + b[j]*dx + c[j]*dx^2 + d[j]*dx^3
    
    return val
end

evaluar_spline

In [2]:
# Definir datos
x_puntos = [0.0, 1.0, 2.0, 3.0]
y_puntos = sin.(x_puntos)

# Calcular coeficientes
a, b, c, d = spline_cubico_natural(x_puntos, y_puntos)

println("Coeficientes obtenidos:")
println("a: ", a)
println("b: ", b)
println("c: ", c)
println("d: ", d)

# Evaluar en un punto intermedio (x = 1.5)
x_eval = 1.5
val_spline = evaluar_spline(x_eval, x_puntos, a, b, c, d)
val_real = sin(x_eval)

println("\n--- Evaluación en x = 1.5 ---")
println("Valor Spline: $val_spline")
println("Valor Real:   $val_real")
println("Error:        $(abs(val_spline - val_real))")

Coeficientes obtenidos:
a: [0.0, 0.8414709848078965, 0.9092974268256817]
b: [0.9920426054996861, 0.540327743424317, -0.4254612987199095]
c: [0.0, -0.45171486207536904, -0.5140741800688574]
d: [-0.15057162069178967, -0.020786439331162792, 0.17135806002295248]

--- Evaluación en x = 1.5 ---
Valor Spline: 0.9961078360848172
Valor Real:   0.9974949866040544
Error:        0.0013871505192372124
